In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "helios_project"
REPO_DIR = PROJECT_ROOT / "Helios"
INPUT_DIR = PROJECT_ROOT / "inputs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
LOG_DIR = PROJECT_ROOT / "logs"

LOCAL_MODEL_DIR = Path("/content/Helios-Distilled")

for p in [PROJECT_ROOT, INPUT_DIR, OUTPUT_DIR, LOG_DIR, LOCAL_MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT    =", PROJECT_ROOT)
print("REPO_DIR        =", REPO_DIR)
print("INPUT_DIR       =", INPUT_DIR)
print("OUTPUT_DIR      =", OUTPUT_DIR)
print("LOG_DIR         =", LOG_DIR)
print("LOCAL_MODEL_DIR =", LOCAL_MODEL_DIR)

PROJECT_ROOT    = /content/drive/MyDrive/helios_project
REPO_DIR        = /content/drive/MyDrive/helios_project/Helios
INPUT_DIR       = /content/drive/MyDrive/helios_project/inputs
OUTPUT_DIR      = /content/drive/MyDrive/helios_project/outputs
LOG_DIR         = /content/drive/MyDrive/helios_project/logs
LOCAL_MODEL_DIR = /content/Helios-Distilled


In [12]:
!nvidia-smi

Mon Apr 13 00:39:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
if not REPO_DIR.exists():
    !git clone https://github.com/PKU-YuanGroup/Helios.git "{REPO_DIR}"
else:
    print("Repo already exists, pulling latest changes...")
    %cd "{REPO_DIR}"
    !git pull

Repo already exists, pulling latest changes...
/content/drive/MyDrive/helios_project/Helios
Already up to date.


In [14]:
%cd "{REPO_DIR}"
!pwd
!ls

/content/drive/MyDrive/helios_project/Helios
/content/drive/MyDrive/helios_project/Helios
app.py	 infer_helios.py  requirements_npu.txt	train_helios.py
eval	 install.sh	  requirements.txt
example  LICENSE.txt	  scripts
helios	 README.md	  tools


In [15]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu130
# Change based on cuda version
!bash install.sh


Looking in indexes: https://download.pytorch.org/whl/cu130
  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-v4gbo0jx
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-v4gbo0jx
  Resolved https://github.com/huggingface/diffusers.git to commit dc8d9032171c83741fd37ed2b12bc9d8274464f3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (645.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.19.1 whic

In [16]:
!ls scripts/inference
!sed -n '1,20p' scripts/inference/helios-distilled_i2v.sh

experiment_interactive	helios-distilled_i2v.sh  helios-mid_t2v.sh
helios-base_i2v.sh	helios-distilled_t2v.sh  helios-mid_v2v.sh
helios-base_t2v.sh	helios-distilled_v2v.sh
helios-base_v2v.sh	helios-mid_i2v.sh
# Example: Running inference with 2-GPU parallelism
# CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node 2 infer_helios.py \
#     --enable_parallelism \
#     --cp_backend "ulysses" \   #  ["ring", "ulysses", "unified", "ulysses_anything"]

CUDA_VISIBLE_DEVICES=0 python infer_helios.py \
    --base_model_path "BestWishYsh/Helios-Distilled" \
    --transformer_path "BestWishYsh/Helios-Distilled" \
    --sample_type "i2v" \
    --num_frames 240 \
    --fps 24 \
    --image_path "example/wave.jpg" \
    --image_noise_sigma_min 0.111 \
    --image_noise_sigma_max 0.135 \
    --prompt "A towering emerald wave surges forward, its crest curling with raw power and energy. Sunlight glints off the translucent water, illuminating the intricate textures and deep green hues within the wave’s b

In [17]:
from pathlib import Path

LOCAL_MODEL_DIR = Path("/content/Helios-Distilled")
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(LOCAL_MODEL_DIR)

/content/Helios-Distilled


In [18]:
!du -sh /content/Helios-Distilled
!find /content/Helios-Distilled -maxdepth 1 | head -30

4.0K	/content/Helios-Distilled
/content/Helios-Distilled


In [20]:
!pip install -U "huggingface_hub[cli]"
!hf download BestWishYSH/Helios-Distilled --local-dir /content/Helios-Distilled

Fetching 35 files:   0% 0/35 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 35 files: 100% 35/35 [06:03<00:00, 10.38s/it]
Download complete: : 138GB [06:03, 402MB/s]                             /content/Helios-Distilled
Download complete: : 138GB [06:03, 379MB/s]


In [21]:
!du -sh /content/Helios-Distilled
!find /content/Helios-Distilled -maxdepth 2 | head -50

129G	/content/Helios-Distilled
/content/Helios-Distilled
/content/Helios-Distilled/.gitattributes
/content/Helios-Distilled/tokenizer
/content/Helios-Distilled/tokenizer/special_tokens_map.json
/content/Helios-Distilled/tokenizer/tokenizer.json
/content/Helios-Distilled/tokenizer/spiece.model
/content/Helios-Distilled/tokenizer/tokenizer_config.json
/content/Helios-Distilled/model_index.json
/content/Helios-Distilled/scheduler
/content/Helios-Distilled/scheduler/scheduler_config.json
/content/Helios-Distilled/text_encoder
/content/Helios-Distilled/text_encoder/model-00005-of-00005.safetensors
/content/Helios-Distilled/text_encoder/model-00001-of-00005.safetensors
/content/Helios-Distilled/text_encoder/model-00004-of-00005.safetensors
/content/Helios-Distilled/text_encoder/model-00003-of-00005.safetensors
/content/Helios-Distilled/text_encoder/model-00002-of-00005.safetensors
/content/Helios-Distilled/text_encoder/config.json
/content/Helios-Distilled/text_encoder/model.safetensors.inde

In [22]:
!wget -O "{INPUT_DIR / 'wave.jpg'}" "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/helios/wave.jpg"
!ls "{INPUT_DIR}"

--2026-04-13 00:54:48--  https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/helios/wave.jpg
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.34, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/621ffdd236468d709f1835cf/9ef967734728251c9bf1854b654ed2cec9ca8c69eb36b3f9aebab53e5aa2e147?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wave.jpg%3B+filename%3D%22wave.jpg%22%3B&response-content-type=image%2Fjpeg&Expires=1776045288&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzc2MDQ1Mjg4fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjIxZmZkZDIzNjQ2OGQ3MDlmMTgzNWNmLzllZjk2NzczNDcyODI1MWM5YmYxODU0YjY1NGVkMmNlYzljYThjNjllYjM2YjNmOWFlYmFiNTNlNWFhMmUxNDdcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC1

In [23]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "helios_project"
REPO_DIR = PROJECT_ROOT / "Helios"

INPUT_DIR = PROJECT_ROOT / "inputs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
LOG_DIR = PROJECT_ROOT / "logs"
RESULT_DIR = PROJECT_ROOT / "results"

MODEL_DIR = Path("/content/Helios-Distilled")

for p in [INPUT_DIR, OUTPUT_DIR, LOG_DIR, RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("REPO_DIR   =", REPO_DIR)
print("INPUT_DIR  =", INPUT_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("MODEL_DIR  =", MODEL_DIR)

REPO_DIR   = /content/drive/MyDrive/helios_project/Helios
INPUT_DIR  = /content/drive/MyDrive/helios_project/inputs
OUTPUT_DIR = /content/drive/MyDrive/helios_project/outputs
MODEL_DIR  = /content/Helios-Distilled


In [24]:
!ls -lah /content/Helios-Distilled | head -30
!ls -lah /content/drive/MyDrive/helios_project/inputs

total 84K
drwxr-xr-x 10 root root 4.0K Apr 13 00:50 .
drwxr-xr-x  1 root root 4.0K Apr 13 00:44 ..
drwxr-xr-x  3 root root 4.0K Apr 13 00:44 .cache
-rw-r--r--  1 root root 2.2K Apr 13 00:44 .gitattributes
drwxr-xr-x  2 root root 4.0K Apr 13 00:44 guider
-rw-r--r--  1 root root  461 Apr 13 00:44 model_index.json
-rw-r--r--  1 root root 1.8K Apr 13 00:44 modular_model_index.json
-rw-r--r--  1 root root  32K Apr 13 00:44 README.md
drwxr-xr-x  2 root root 4.0K Apr 13 00:44 scheduler
drwxr-xr-x  2 root root 4.0K Apr 13 00:45 text_encoder
drwxr-xr-x  2 root root 4.0K Apr 13 00:44 tokenizer
drwxr-xr-x  2 root root 4.0K Apr 13 00:50 transformer
drwxr-xr-x  2 root root 4.0K Apr 13 00:50 transformer_ode
drwxr-xr-x  2 root root 4.0K Apr 13 00:48 vae
total 7.7M
-rw------- 1 root root 7.7M Apr 13 00:54 wave.jpg


In [25]:
%cd "{REPO_DIR}"
!sed -n '1,220p' scripts/inference/helios-distilled_i2v.sh

/content/drive/MyDrive/helios_project/Helios
# Example: Running inference with 2-GPU parallelism
# CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node 2 infer_helios.py \
#     --enable_parallelism \
#     --cp_backend "ulysses" \   #  ["ring", "ulysses", "unified", "ulysses_anything"]

CUDA_VISIBLE_DEVICES=0 python infer_helios.py \
    --base_model_path "BestWishYsh/Helios-Distilled" \
    --transformer_path "BestWishYsh/Helios-Distilled" \
    --sample_type "i2v" \
    --num_frames 240 \
    --fps 24 \
    --image_path "example/wave.jpg" \
    --image_noise_sigma_min 0.111 \
    --image_noise_sigma_max 0.135 \
    --prompt "A towering emerald wave surges forward, its crest curling with raw power and energy. Sunlight glints off the translucent water, illuminating the intricate textures and deep green hues within the wave’s body. A thick spray erupts from the breaking crest, casting a misty veil that dances above the churning surface. As the perspective widens, the immense scale of the

In [26]:
from pathlib import Path

run1_dir = OUTPUT_DIR / "baseline_i2v_run1"
run1_dir.mkdir(parents=True, exist_ok=True)

image_path = INPUT_DIR / "wave.jpg"
prompt = (
    "A towering emerald wave surges forward immediately from the beginning. "
    "The crest curls with visible motion, water spray bursts outward, and the ocean moves naturally."
)

cmd = f"""
cd "{REPO_DIR}" && \
CUDA_VISIBLE_DEVICES=0 python infer_helios.py \
  --base_model_path "{MODEL_DIR}" \
  --transformer_path "{MODEL_DIR}" \
  --sample_type "i2v" \
  --image_path "{image_path}" \
  --prompt "{prompt}" \
  --height 384 \
  --width 640 \
  --num_frames 99 \
  --guidance_scale 1.0 \
  --is_enable_stage2 \
  --pyramid_num_inference_steps_list 2 2 2 \
  --is_amplify_first_chunk \
  --output_folder "{run1_dir}"
"""

print(cmd)
!bash -lc '{cmd}'


cd "/content/drive/MyDrive/helios_project/Helios" && CUDA_VISIBLE_DEVICES=0 python infer_helios.py   --base_model_path "/content/Helios-Distilled"   --transformer_path "/content/Helios-Distilled"   --sample_type "i2v"   --image_path "/content/drive/MyDrive/helios_project/inputs/wave.jpg"   --prompt "A towering emerald wave surges forward immediately from the beginning. The crest curls with visible motion, water spray bursts outward, and the ocean moves naturally."   --height 384   --width 640   --num_frames 99   --guidance_scale 1.0   --is_enable_stage2   --pyramid_num_inference_steps_list 2 2 2   --is_amplify_first_chunk   --output_folder "/content/drive/MyDrive/helios_project/outputs/baseline_i2v_run1"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version 

In [27]:
!find "{run1_dir}" -maxdepth 3 -type f | sort

/content/drive/MyDrive/helios_project/outputs/baseline_i2v_run1/0000_i2v_1776042317.mp4


In [28]:
from IPython.display import Video, display
import glob

videos = sorted(glob.glob(str(run1_dir / "*.mp4")))
print(videos)

if videos:
    display(Video(videos[0], embed=True))
else:
    print("No mp4 found in run1_dir.")

['/content/drive/MyDrive/helios_project/outputs/baseline_i2v_run1/0000_i2v_1776042317.mp4']


In [29]:
run_no_amp = OUTPUT_DIR / "baseline_no_amplify"
run_no_amp.mkdir(parents=True, exist_ok=True)

cmd = f"""
cd "{REPO_DIR}" && \
CUDA_VISIBLE_DEVICES=0 python infer_helios.py \
  --base_model_path "{MODEL_DIR}" \
  --transformer_path "{MODEL_DIR}" \
  --sample_type "i2v" \
  --image_path "{image_path}" \
  --prompt "{prompt}" \
  --height 384 \
  --width 640 \
  --num_frames 99 \
  --guidance_scale 1.0 \
  --is_enable_stage2 \
  --pyramid_num_inference_steps_list 2 2 2 \
  --output_folder "{run_no_amp}"
"""
!bash -lc '{cmd}'

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Fetching 22 files: 100% 22/22 [00:00<00:00, 130700.69it/s]
Download complete: : 0.00B [00:00, ?B/s]
Flash Attn 2 is installed!
Sage Attn is not installed!
Xformers is not installed!
Loading checkpoint shards: 100% 6/6 [00:04<00:00,  1.22it/s]
Patched 160 FP32_RMSNorm modules

Patched 40 Flash_LayerNorm modules

Patched 160 Flash_RMSNorm modules

Patched Flash_RoPE globally

Attention backends are an experimental feature and the API may be subject to change.
Fetching 22 files: 100% 22/22 [00:00<00:00, 25469.14it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading checkpoint shards: 100% 1/1 [00:00<00:00, 119.97it/s]
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]
Loading weights:   0%

In [30]:
videos = sorted(glob.glob(str(run_no_amp / "*.mp4")))
print(videos)

if videos:
    display(Video(videos[0], embed=True))
else:
    print("No mp4 found in run_no_amp.")

['/content/drive/MyDrive/helios_project/outputs/baseline_no_amplify/0000_i2v_1776042791.mp4']


In [31]:
run_skip = OUTPUT_DIR / "baseline_skip_chunk"
run_skip.mkdir(parents=True, exist_ok=True)

cmd = f"""
cd "{REPO_DIR}" && \
CUDA_VISIBLE_DEVICES=0 python infer_helios.py \
  --base_model_path "{MODEL_DIR}" \
  --transformer_path "{MODEL_DIR}" \
  --sample_type "i2v" \
  --image_path "{image_path}" \
  --prompt "{prompt}" \
  --height 384 \
  --width 640 \
  --num_frames 99 \
  --guidance_scale 1.0 \
  --is_enable_stage2 \
  --pyramid_num_inference_steps_list 2 2 2 \
  --is_skip_first_chunk \
  --output_folder "{run_skip}"
"""
!bash -lc '{cmd}'

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Fetching 22 files: 100% 22/22 [00:00<00:00, 18990.47it/s]
Download complete: : 0.00B [00:00, ?B/s]
Flash Attn 2 is installed!
Sage Attn is not installed!
Xformers is not installed!
Loading checkpoint shards: 100% 6/6 [00:03<00:00,  1.67it/s]
Patched 160 FP32_RMSNorm modules

Patched 40 Flash_LayerNorm modules

Patched 160 Flash_RMSNorm modules

Patched Flash_RoPE globally

Attention backends are an experimental feature and the API may be subject to change.
Fetching 22 files: 100% 22/22 [00:00<00:00, 176771.43it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading checkpoint shards: 100% 1/1 [00:00<00:00, 119.09it/s]
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]
Loading weights:   0%

In [32]:
videos = sorted(glob.glob(str(run_skip / "*.mp4")))
print(videos)

if videos:
    display(Video(videos[0], embed=True))
else:
    print("No mp4 found in run_skip.")

['/content/drive/MyDrive/helios_project/outputs/baseline_skip_chunk/0000_i2v_1776042924.mp4']


In [33]:
run_noise = OUTPUT_DIR / "baseline_noise"
run_noise.mkdir(parents=True, exist_ok=True)

cmd = f"""
cd "{REPO_DIR}" && \
CUDA_VISIBLE_DEVICES=0 python infer_helios.py \
  --base_model_path "{MODEL_DIR}" \
  --transformer_path "{MODEL_DIR}" \
  --sample_type "i2v" \
  --image_path "{image_path}" \
  --prompt "{prompt}" \
  --height 384 \
  --width 640 \
  --num_frames 99 \
  --guidance_scale 1.0 \
  --is_enable_stage2 \
  --pyramid_num_inference_steps_list 2 2 2 \
  --image_noise_sigma_min 0.13 \
  --image_noise_sigma_max 0.16 \
  --video_noise_sigma_min 0.13 \
  --video_noise_sigma_max 0.16 \
  --output_folder "{run_noise}"
"""
!bash -lc '{cmd}'

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Fetching 22 files: 100% 22/22 [00:00<00:00, 21434.31it/s]
Download complete: : 0.00B [00:00, ?B/s]
Flash Attn 2 is installed!
Sage Attn is not installed!
Xformers is not installed!
Loading checkpoint shards: 100% 6/6 [00:03<00:00,  1.70it/s]
Patched 160 FP32_RMSNorm modules

Patched 40 Flash_LayerNorm modules

Patched 160 Flash_RMSNorm modules

Patched Flash_RoPE globally

Attention backends are an experimental feature and the API may be subject to change.
Fetching 22 files: 100% 22/22 [00:00<00:00, 17390.63it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading checkpoint shards: 100% 1/1 [00:00<00:00, 119.32it/s]
Loading pipeline components...:  40% 2/5 [00:02<00:03,  1.07s/it]
Loading weight

In [34]:
videos = sorted(glob.glob(str(run_noise / "*.mp4")))
print(videos)

if videos:
    display(Video(videos[0], embed=True))
else:
    print("No mp4 found in run_noise.")

['/content/drive/MyDrive/helios_project/outputs/baseline_noise/0000_i2v_1776043026.mp4']
